# End-to-End Competition Workflow — From Notebook to Kaggle Submission

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb18_competition_workflow_student.ipynb)


## Learning Objectives

By the end of this notebook, you will be able to:

1. Walk a binary classification problem end-to-end through the eight workflow steps the case competition expects.
2. Refactor a notebook's modeling code into reusable functions (`build_pipeline`, `train_pipeline`, `predict_pipeline`).
3. Save and reload the trained pipeline with `joblib` so a teammate can reproduce predictions on their machine.
4. Generate a `submission.csv` file in the exact column format the Kaggle competition requires.
5. Run the full pipeline on the locked competition test set **once** at the end — the Day 20 final-submission ceremony.


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit. The second exercise produces the `submission.csv` you upload to the Kaggle leaderboard.


## 💼 Why This Matters: Two Days from the Competition Deadline

The **Bank Churn case competition** locks the leaderboard on **Day 20 at 11:59 PM**. The team lead at **RetailBank Customer Retention** has approved your improved model from Milestone 3 and now needs the artifacts a real production handoff requires:

> *"I want one notebook that runs end-to-end, a serialized pipeline a new analyst can load on their machine, and a `submission.csv` that matches the Kaggle column spec exactly. If anything in that chain breaks, we lose 48 hours of competition time."*

This notebook is the production-pipeline blueprint. Sections 1–6 walk the eight workflow steps the competition rubric expects (setup, load, EDA, preprocessing, baseline model, improved model, tuning, evaluation). Section 7 refactors the working code into three reusable functions. Section 8 saves the trained pipeline to disk with `joblib`. Section 9 loads it back, runs predictions on the locked Kaggle test set, and writes `submission.csv` with the exact columns the leaderboard expects.

**A question that often comes up here:** *"Why are we touching `X_test` in this notebook when CLAUDE.md says we shouldn't?"* The Kaggle test set is **not** a model-evaluation set — it has no labels, so we cannot score it. We only call `pipeline.predict_proba(X_test)` to **produce predictions for submission**. That is the production-pipeline pattern (a model that runs on unlabeled new data), not model evaluation. The CV-first audit (`scripts/audit_cv_first.py`) recognizes this notebook's submission cell as the one acceptable exception alongside nb14 cell 33.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")


**Reading the output:** The new tools today are `joblib` (pipeline persistence) and `Path` (artifact paths). Everything else is workflow furniture you have used since nb02.


## 2. Load the Competition Data and Sanity Checks

The case-competition starter pack supplies `train.csv` (labeled) and `test.csv` (unlabeled — the leaderboard scorer holds the labels). Section 9 only opens the test file at submission time; for sections 2–8 we work strictly on the labeled training data.

> 💡 **Gemini Prompt:** "Load `train.csv` and `test.csv` from the case-competition data folder. Print shapes, dtypes, missing-value counts, and the target class balance for `train.csv`. The target column is `Exited`."
>
> **After running, verify:**
> - Both files load without dtype warnings (or, if they warn, the warning is named in your README).
> - The target balance is around 0.20 — the dataset is moderately imbalanced.


In [ ]:
# Replace the path below with the actual case-competition data location.
DATA_DIR = Path("../_course_case_competition/2026Summer/reference")  # adjust as needed
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET = "Exited"

if TRAIN_PATH.exists():
    df_train_full = pd.read_csv(TRAIN_PATH)
    df_test_kaggle = pd.read_csv(TEST_PATH)
    print(f"Train: {df_train_full.shape}")
    print(f"Test (no labels): {df_test_kaggle.shape}")
    print(f"Class balance (train): {df_train_full[TARGET].mean():.3f}")
else:
    print("Competition data not found at the path above.")
    print("Update DATA_DIR to point to your local checkout of train.csv / test.csv.")


**Reading the output:**

If the load succeeds you should see two row counts and a class-balance number near 0.20. If the path is wrong, fix `DATA_DIR` and rerun — the rest of the notebook depends on `df_train_full` and `df_test_kaggle` being defined.

**A question that often comes up here:** *"What if the train and test files have different columns?"* The Kaggle test file should have every feature column from `train.csv` minus the target column (which the scorer holds). Run `set(df_train_full.columns) - set(df_test_kaggle.columns)` — the only difference should be `{"Exited"}`. Anything else means your starter pack is wrong; flag it on the course Slack before continuing.


## 3. EDA Snapshot

Three plots — class balance, missingness, top-correlated numeric features — are enough to confirm the dataset matches the rubric's description and to identify any column you should drop or transform.

> 💡 **Gemini Prompt:** "From `df_train_full`, plot: (a) `Exited` value counts as a bar chart; (b) per-column missing-value count as a bar chart, sorted descending; (c) absolute correlation of every numeric column with `Exited`, top 10. One figure with 1x3 subplots."


In [ ]:
# YOUR EDA SNAPSHOT CODE HERE

# Hints:
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# df_train_full[TARGET].value_counts().plot.bar(ax=axes[0], title="Class balance")
# df_train_full.isna().sum().sort_values(ascending=False).head(10).plot.bar(ax=axes[1], title="Missingness")
# corrs = df_train_full.corr(numeric_only=True)[TARGET].abs().sort_values(ascending=False).head(10)
# corrs.plot.bar(ax=axes[2], title=f"|corr| with {TARGET}")
# plt.tight_layout(); plt.show()


## 4. Preprocessing Pipeline

Wrap every column transformation inside a `ColumnTransformer` so the same logic that fits on training data is the logic that predicts on the Kaggle test set. This is the single most common production failure: training-time and inference-time preprocessing diverge, and predictions look wrong on Kaggle even though CV scores looked great.


In [ ]:
def build_preprocessor(df, target=TARGET):
    feature_df = df.drop(columns=[target]) if target in df.columns else df
    numeric_cols = feature_df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = feature_df.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    pre = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])
    return pre, numeric_cols, categorical_cols

if 'df_train_full' in globals():
    pre, numeric_cols, categorical_cols = build_preprocessor(df_train_full)
    print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols[:8]}{' ...' if len(numeric_cols) > 8 else ''}")
    print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


**Reading the output:**

Two lists, one numeric, one categorical, that together account for every column except the target. If the printout shows zero categorical columns and you expected at least `Geography` or `Gender`, double-check `df_train_full.dtypes` — `pd.read_csv` sometimes infers `int64` for ID columns that should be excluded.


## 5. Baseline Model — Logistic Regression


In [ ]:
def fit_baseline(df, pre, target=TARGET):
    X = df.drop(columns=[target])
    y = df[target]
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    aucs = cross_val_score(pipe, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
    pipe.fit(X, y)
    return pipe, aucs

if 'df_train_full' in globals():
    baseline_pipe, baseline_aucs = fit_baseline(df_train_full, pre)
    print(f"Baseline logistic CV ROC-AUC: {baseline_aucs.mean():.3f} ± {baseline_aucs.std(ddof=1):.3f}")


**Reading the output:**

A baseline AUC near 0.75–0.80 on the Bank Churn data is the expected anchor — the improved model in the next section needs to beat this with non-overlapping confidence intervals to be worth promoting. If your baseline AUC is dramatically different, walk back through sections 2–4 to confirm the preprocessor matched the data shape correctly.


## 6. Improved Model — Gradient Boosting


In [ ]:
def fit_improved(df, pre, target=TARGET):
    X = df.drop(columns=[target])
    y = df[target]
    pipe = Pipeline([
        ("pre", pre),
        ("clf", GradientBoostingClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED,
        )),
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    aucs = cross_val_score(pipe, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
    pipe.fit(X, y)
    return pipe, aucs

if 'df_train_full' in globals():
    improved_pipe, improved_aucs = fit_improved(df_train_full, pre)
    cv_summary = pd.DataFrame({
        "mean": [baseline_aucs.mean(), improved_aucs.mean()],
        "sd":   [baseline_aucs.std(ddof=1), improved_aucs.std(ddof=1)],
    }, index=["baseline_logistic", "improved_gbm"])
    cv_summary["95%CI_halfwidth"] = cv_summary["sd"] / np.sqrt(5) * 2.776
    print(cv_summary)


**Reading the output:**

If the GBM mean exceeds the baseline mean by more than the sum of their CI half-widths, the improved model is the clear champion. If the CIs overlap, the simpler baseline wins by Occam's razor. The competition rubric rewards calibrated, well-tuned ensembles — but only when the CV evidence supports them. **Do not promote a model whose CV CI overlaps the baseline's just because its leaderboard score is slightly higher; that is overfitting to the public leaderboard.**


## 📝 PAUSE-AND-DO Exercise 1 — Pick Your Champion (10 minutes)

**Task:** Decide which of `baseline_pipe` or `improved_pipe` is your competition champion, and justify in three sentences using the CI comparison from the table above.

### YOUR CHAMPION DECISION HERE:

**Champion model:** *[baseline_logistic / improved_gbm]*

**Justification (3 sentences):**

1. *[CV mean comparison]*
2. *[CI overlap or separation — does it support promotion?]*
3. *[business rationale — interpretability, deployment cost, calibration concerns]*


In [ ]:
# Bind your champion choice for the rest of the notebook
champion_pipe = improved_pipe  # or baseline_pipe — change to match your decision


## 7. Refactor — `train_pipeline` and `predict_pipeline`

Two small functions wrap everything above into a clean train/predict interface. A teammate or future-you can call these without rereading the whole notebook.


In [ ]:
def train_pipeline(df, target=TARGET, choice="improved"):
    pre, _, _ = build_preprocessor(df, target=target)
    if choice == "improved":
        clf = GradientBoostingClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED,
        )
    else:
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(df.drop(columns=[target]), df[target])
    return pipe

def predict_pipeline(pipe, df):
    return pipe.predict_proba(df)[:, 1]

if 'df_train_full' in globals():
    refit = train_pipeline(df_train_full, choice="improved")
    refit_aucs = cross_val_score(
        refit, df_train_full.drop(columns=[TARGET]), df_train_full[TARGET],
        scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED),
        n_jobs=-1,
    )
    print(f"Refit champion CV ROC-AUC: {refit_aucs.mean():.3f} ± {refit_aucs.std(ddof=1):.3f}")


**Reading the output:**

The refit's CV AUC should match the section-6 number to three decimals. If it does not, the refactor changed something silently — usually a feature column was dropped or recast. Diff the two pipelines (`pipe.named_steps`) until the discrepancy is resolved before saving.


## 8. Save the Trained Pipeline with `joblib`

`joblib.dump` writes the entire fitted pipeline (preprocessing + estimator + learned parameters) to a single `.joblib` file. A teammate can `joblib.load` that file and call `predict_proba` without rerunning training.


In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
ARTIFACT_PATH = ARTIFACT_DIR / "champion_pipeline.joblib"

joblib.dump(refit, ARTIFACT_PATH)
print(f"Pipeline saved -> {ARTIFACT_PATH}")
print(f"Size on disk  : {ARTIFACT_PATH.stat().st_size / 1024:.1f} KB")


**Reading the output:**

The artifact path and a size in kilobytes (or a few megabytes for tree ensembles) confirm the pipeline serialized cleanly. Add this `.joblib` file to your project's `artifacts/` directory and reference it in your M4 poster's "Reproducibility" line — graders should be able to load it and reproduce predictions in under 30 seconds.


## 9. Generate `submission.csv` for Kaggle

Load the saved pipeline back from disk (the same way a grader would) and run it on the locked Kaggle test file. The `submission.csv` must have **exactly two columns** in the order the leaderboard scorer expects: `id` (the row identifier) and the predicted probability under the column name the competition specifies (often `Exited`).

> 💡 **Gemini Prompt:** "Load the joblib pipeline from `ARTIFACT_PATH`. Predict `predict_proba` on `df_test_kaggle`. Build a DataFrame with columns `[id, Exited]` where `id` comes from `df_test_kaggle['id']` (or whatever ID column the competition uses) and `Exited` is the predicted probability. Save as `submission.csv` with `index=False`."


In [ ]:
# YOUR SUBMISSION CODE HERE

# Hints:
# loaded = joblib.load(ARTIFACT_PATH)
# X_test_kaggle = df_test_kaggle.drop(columns=["id"])  # adjust if id col differs
# proba = loaded.predict_proba(X_test_kaggle)[:, 1]
# submission = pd.DataFrame({"id": df_test_kaggle["id"], "Exited": proba})
# submission.to_csv("submission.csv", index=False)
# print(submission.head())


## 📝 PAUSE-AND-DO Exercise 2 — File the Submission (10 minutes)

**Task:** Generate `submission.csv` and confirm it is leaderboard-ready.

**Checklist:**

- [ ] `submission.csv` has the exact column names the competition page lists (case-sensitive).
- [ ] Row count matches `df_test_kaggle` row count.
- [ ] `Exited` (or the predicted-probability column) is a float in `[0, 1]`, not 0/1.
- [ ] No `NaN` rows — `submission.isna().sum().sum() == 0`.
- [ ] First three rows look like `(id, probability)` pairs you would expect.

Once those five checks pass, upload to the Kaggle leaderboard.


## 10. Wrap-Up — Key Takeaways

1. **A production pipeline is one object, not two.** Preprocessing and modeling live inside the same `Pipeline`, fit together, saved together, loaded together. Splitting them is the most common cause of "works on my machine, fails on Kaggle."
2. **Refactor before you save.** A `.joblib` file is only as good as the code that wrote it. Functions (`train_pipeline`, `predict_pipeline`) are easier to verify than a 200-cell notebook.
3. **The submission file is a contract.** Two columns, exact names, no `NaN`. The leaderboard scorer is a strict parser, not a forgiving teammate.
4. **The Kaggle test set is unlabeled — that is what makes calling `.predict_proba(X_test)` here OK.** This is a production prediction, not a model evaluation. The CV-first audit's nb18 exception is for exactly this pattern.

**A question that often comes up here:** *"What if my Kaggle score is much worse than my CV mean?"* Three usual culprits, in order of frequency. (1) The train/test files have different column distributions ("covariate shift") — compare numeric column means and categorical counts. (2) Your preprocessing dropped or transformed a column on training data that is missing on test data — `pipe.named_steps["pre"].transform(df_test_kaggle).shape` should match the training transform. (3) You are overfitting to your CV folds because of a leaky feature; check whether any feature is too good to be true.

**Next stop — nb19: Special Topic — Deep Learning.** With a working pipeline shipped to the leaderboard, nb19 widens the lens: what changes when the data are images or sequences? What is a neural network, why are convolutions and recurrence the two structural inventions that made deep learning work, and when is deep learning **not** the right tool for a tabular business problem?


## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises**.
2. **Run all cells**.
3. **Save with output** and submit `nb18_competition_workflow_<your_lastname>.ipynb` to Brightspace.
4. **Upload `submission.csv`** to the Kaggle leaderboard before Day 20 11:59 PM.

**Bibliography**
- Chip Huyen: *Designing Machine Learning Systems* (chapters on serving and pipelines).
- scikit-learn User Guide: model persistence with `joblib`.
- Course case-competition starter pack: `_course_case_competition/2026Summer/`.


<center>

# Thank you!

</center>
